# ABA Transaction Analysis
## Step 1: Parse result.json → DataFrame

In [19]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [4]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt

with open("telegram_aba_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

pattern = re.compile(
    r"\$([\d.]+) paid by (.+?) \(\*\d+\) on .+? via (.+?) at \d+s by S\.C\. Trx\. ID: (\d+)"
)

rows = []
for msg in data["messages"]:
    if msg.get("type") != "message":
        continue
    text = msg.get("text", "")
    if not isinstance(text, str):
        continue
    match = pattern.search(text)
    if match:
        amount, name, method, trx_id = match.groups()
        rows.append({
            "Amount": float(amount),
            "Customer_Name": name.strip(),
            "DateTime": msg.get("date"),
            "Payment_Method": method.strip(),
            "Transaction_ID": trx_id
        })

df = pd.DataFrame(rows)
df['DateTime'] = pd.to_datetime(df['DateTime'])
df['Date'] = df['DateTime'].dt.date
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.day_name()
df['Month'] = df['DateTime'].dt.to_period('M').astype(str)

print(f"Loaded {len(df)} transactions")
df.head()

Loaded 5509 transactions


,Amount,Customer_Name,DateTime,Payment_Method,Transaction_ID,Date,Hour,DayOfWeek,Month
0,4.5,HALIM,2025-10-02 12:57:35,ABA PAY,175938465476344,2025-10-02,12,Thursday,2025-10
1,17.0,emul maulana,2025-10-02 14:35:01,ABA KHQR (ACLEDA Bank Plc.),175939050053954,2025-10-02,14,Thursday,2025-10
2,12.0,DODI JORDI,2025-10-02 14:41:08,ABA KHQR (Wing Bank (Cambodia) Plc),175939086845771,2025-10-02,14,Thursday,2025-10
3,19.0,RICKO TANOTO,2025-10-02 14:48:34,ABA KHQR (Wing Bank (Cambodia) Plc),175939131415162,2025-10-02,14,Thursday,2025-10
4,13.0,PAULINUS LAIA,2025-10-02 14:54:16,ABA KHQR (Wing Bank (Cambodia) Plc),175939165617183,2025-10-02,14,Thursday,2025-10


## Step 2: Dataset Overview

In [5]:
print(f"Total transactions : {len(df)}")
print(f"Total revenue      : ${df['Amount'].sum():,.2f}")
print(f"Average per txn    : ${df['Amount'].mean():,.2f}")
print(f"Date range         : {df['DateTime'].min().date()} to {df['DateTime'].max().date()}")
print(f"Unique customers   : {df['Customer_Name'].nunique()}")
print(f"Payment methods    : {df['Payment_Method'].nunique()}")

Total transactions : 5509
Total revenue      : $79,347.90
Average per txn    : $14.40
Date range         : 2025-10-02 to 2026-06-16
Unique customers   : 3678
Payment methods    : 48


## Step 3: Top Customers by Frequency (Who Buys A Lot)

In [22]:
top_freq = (df.groupby('Customer_Name')
            .agg(Transactions=('Amount', 'count'),
                 Total_Spent=('Amount', 'sum'))
            .sort_values('Transactions', ascending=True)
            .tail(15).reset_index())

fig = px.bar(top_freq, x='Transactions', y='Customer_Name', orientation='h',
             color='Total_Spent', color_continuous_scale='Blues',
             title='Top 15 Repeat Customers',
             hover_data=['Total_Spent'])
fig.update_layout(yaxis_title='', height=500)
fig.show()

# ========== TOP 15 BY SPENDING ==========

In [23]:
top_spend = (df.groupby('Customer_Name')['Amount']
             .sum().sort_values(ascending=True)
             .tail(15).reset_index())

fig = px.bar(top_spend, x='Amount', y='Customer_Name', orientation='h',
             color='Amount', color_continuous_scale='Greens',
             title='Top 15 Customers by Total Spending')
fig.update_layout(yaxis_title='', height=500)
fig.show()

# ========== PAYMENT METHOD ==========

In [24]:
method_stats = df.groupby('Method_Clean')['Amount'].agg(['count','sum']).reset_index()
method_stats.columns = ['Method', 'Count', 'Total']

fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"pie"}]],
                    subplot_titles=['By Transaction Count','By Revenue'])
fig.add_trace(go.Pie(labels=method_stats['Method'], values=method_stats['Count']), row=1, col=1)
fig.add_trace(go.Pie(labels=method_stats['Method'], values=method_stats['Total']), row=1, col=2)
fig.update_layout(title='Payment Method Breakdown', height=400)
fig.show()

# ========== DAILY REVENUE TREND ==========

In [25]:
daily = df.groupby('Date')['Amount'].sum().reset_index()

fig = px.line(daily, x='Date', y='Amount',
              title='Daily Revenue Trend')
fig.update_layout(yaxis_title='Revenue ($)', xaxis_title='')
fig.show()

# ========== TRANSACTIONS BY HOUR ==========

In [26]:
hourly = df.groupby('Hour')['Amount'].count().reset_index()
hourly.columns = ['Hour', 'Count']

fig = px.bar(hourly, x='Hour', y='Count',
             title='Transactions by Hour',
             color='Count', color_continuous_scale='Oranges')
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

# ========== TRANSACTIONS BY DAY OF WEEK ==========

In [27]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily_dow = df.groupby('DayOfWeek')['Amount'].count().reindex(day_order).reset_index()
daily_dow.columns = ['Day', 'Count']

fig = px.bar(daily_dow, x='Day', y='Count',
             title='Transactions by Day of Week',
             color='Count', color_continuous_scale='Purples')
fig.show()

# ========== AMOUNT DISTRIBUTION ==========

In [28]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Distribution (Under $50)', 'Full Range Box Plot'])

fig.add_trace(go.Histogram(x=df[df['Amount']<=50]['Amount'], nbinsx=25,
              marker_color='#0088cc', name='Amount'), row=1, col=1)
fig.add_trace(go.Box(y=df['Amount'], marker_color='#0088cc',
              name='Amount'), row=1, col=2)
fig.update_layout(title='Transaction Amount Distribution', height=400, showlegend=False)
fig.show()


# ========== MONTHLY REVENUE TREND ==========

In [29]:
monthly = df.groupby('Month').agg(
    Revenue=('Amount','sum'),
    Transactions=('Amount','count')).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly['Month'], y=monthly['Revenue'],
              name='Revenue', marker_color='#0088cc'), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['Month'], y=monthly['Transactions'],
              name='Transactions', marker_color='#ff6600', mode='lines+markers'), secondary_y=True)
fig.update_layout(title='Monthly Revenue & Transaction Count')
fig.update_yaxes(title_text='Revenue ($)', secondary_y=False)
fig.update_yaxes(title_text='Transactions', secondary_y=True)
fig.show()

## Step 9: Monthly Summary

In [17]:
monthly = (df.groupby('Month')
           .agg(Transactions=('Amount', 'count'),
                Revenue=('Amount', 'sum'),
                Avg_Amount=('Amount', 'mean'),
                Unique_Customers=('Customer_Name', 'nunique'))
           .round(2))
monthly

,Transactions,Revenue,Avg_Amount,Unique_Customers
Month,,,,
2025-10,264,6201.90,23.49,158
2025-11,505,8451.85,16.74,356
2025-12,648,11803.06,18.21,454
2026-01,709,10732.52,15.14,454
2026-02,858,11077.01,12.91,692
2026-03,895,10599.23,11.84,800
2026-04,720,8935.10,12.41,655
2026-05,609,7906.29,12.98,559
2026-06,301,3640.94,12.10,274


## Step 10: Export to CSV for Power BI

In [30]:
df.to_csv("aba_transactions.csv", index=False)
print(f"Exported {len(df)} rows to aba_transactions.csv")

Exported 5509 rows to aba_transactions.csv
